# Pyro BNN Surrogate + BoTorch Bayesian Optimization

This notebook demonstrates a custom Bayesian neural-network surrogate inside a Bayesian-optimization loop:

```text
initial experiments → Pyro BNN → posterior function samples → BoTorch qLogEI → next experiment
```

The expensive objective may represent a physical experiment, discrete-event simulation, digital twin, FEA/CFD model, or process trial.


In [ ]:
from typing import Optional
import numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn, pyro
from torch import Tensor
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.nn import PyroModule, PyroSample
from botorch.models.ensemble import EnsembleModel
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim import optimize_acqf

SEED=7
np.random.seed(SEED); torch.manual_seed(SEED); pyro.set_rng_seed(SEED)
torch.set_default_dtype(torch.float64)


## 1. Expensive black-box process

The optimizer does not know this analytical expression; it is used only to generate synthetic experimental observations. The objective is maximized.


In [ ]:
def true_process(X:Tensor)->Tensor:
    x1,x2=X[...,0],X[...,1]
    peak=1.35*torch.exp(-((x1-.68)**2/.025+(x2-.34)**2/.035))
    second=.45*torch.exp(-((x1-.28)**2/.06+(x2-.72)**2/.08))
    ripple=.08*torch.sin(9*x1)*torch.cos(7*x2)
    penalty=.22*x1+.12*x2
    return peak+second+ripple-penalty

def observe(X,noise=.025): return (true_process(X)+noise*torch.randn(X.shape[:-1])).unsqueeze(-1)
bounds=torch.tensor([[0.,0.],[1.,1.]])
train_X=torch.rand(16,2); train_Y=observe(train_X)
print('Initial best observation:',float(train_Y.max()))


## 2. Pyro BNN surrogate


In [ ]:
class ProcessBNN(PyroModule):
    def __init__(self,d=2,h=20):
        super().__init__(); self.h=PyroModule[nn.Linear](d,h); self.o=PyroModule[nn.Linear](h,1)
        self.h.weight=PyroSample(dist.Normal(0,1).expand([h,d]).to_event(2)); self.h.bias=PyroSample(dist.Normal(0,1).expand([h]).to_event(1))
        self.o.weight=PyroSample(dist.Normal(0,1).expand([1,h]).to_event(2)); self.o.bias=PyroSample(dist.Normal(0,1).expand([1]).to_event(1))
    def forward(self,x,y=None):
        mean=self.o(torch.tanh(self.h(x))).squeeze(-1); sigma=pyro.sample('sigma',dist.LogNormal(-3,.45))
        with pyro.plate('data',x.shape[0]): pyro.sample('obs',dist.StudentT(4,mean,sigma),obs=y)
        return mean

def fit(model,guide,X,Y,steps=800):
    svi=SVI(model,guide,pyro.optim.Adam({'lr':.02}),loss=Trace_ELBO())
    return [svi.step(X,Y.squeeze(-1))/len(X) for _ in range(steps)]


## 3. Adapt Pyro posterior samples to BoTorch

BoTorch Monte Carlo acquisition functions can work with non-GP models when the custom model exposes an appropriate posterior. `EnsembleModel` is used here to represent posterior function samples.


In [ ]:
class PyroBNNEnsemble(EnsembleModel):
    _num_outputs=1
    def __init__(self,model,guide,num_samples=128):
        super().__init__(); self.model=model; self.guide=guide; self.num_samples=num_samples
    def forward(self,X):
        # X: batch_shape x q x d
        batch_shape=X.shape[:-2]; q=X.shape[-2]; d=X.shape[-1]; flat=X.reshape(-1,d)
        pred=Predictive(self.model,guide=self.guide,num_samples=self.num_samples,return_sites=('obs',))(flat)['obs']
        # sample x (batch*q) -> batch x sample x q x 1
        shaped=pred.reshape(self.num_samples,*batch_shape,q).movedim(0,len(batch_shape))
        return shaped.unsqueeze(-1)

pyro.clear_param_store(); pyro_model=ProcessBNN(); guide=AutoDiagonalNormal(pyro_model); losses=fit(pyro_model,guide,train_X,train_Y)
bnn=PyroBNNEnsemble(pyro_model,guide,128)
plt.figure(figsize=(8,3)); plt.plot(losses); plt.xlabel('SVI step'); plt.ylabel('ELBO / observation'); plt.show()


## 4. Select the next experiment with qLogEI

Acquisition optimization balances exploitation and exploration through the surrogate posterior.


In [ ]:
acq=qLogExpectedImprovement(model=bnn,best_f=train_Y.max())
candidate,acq_value=optimize_acqf(acq_function=acq,bounds=bounds,q=1,num_restarts=10,raw_samples=128)
new_y=observe(candidate)
print('Suggested process setting:',candidate.detach().numpy().round(4))
print('New observation:',float(new_y))
print('Acquisition value:',float(acq_value))


## 5. Short Bayesian-optimization loop


In [ ]:
X_bo=train_X.clone(); Y_bo=train_Y.clone(); history=[float(Y_bo.max())]
for iteration in range(5):
    pyro.clear_param_store(); pm=ProcessBNN(); g=AutoDiagonalNormal(pm); _=fit(pm,g,X_bo,Y_bo,steps=550); surrogate=PyroBNNEnsemble(pm,g,96)
    acq=qLogExpectedImprovement(model=surrogate,best_f=Y_bo.max())
    cand,_=optimize_acqf(acq_function=acq,bounds=bounds,q=1,num_restarts=8,raw_samples=96)
    y_new=observe(cand); X_bo=torch.cat([X_bo,cand]); Y_bo=torch.cat([Y_bo,y_new]); history.append(float(Y_bo.max()))
    print(f'Iteration {iteration+1}: best={history[-1]:.4f}')

plt.plot(history,marker='o'); plt.xlabel('BO iteration'); plt.ylabel('Best observed objective'); plt.show()


## Takeaway

A BNN can act as a probabilistic surrogate in Bayesian optimization when its posterior can be sampled in a way compatible with the acquisition function. For small expensive datasets, however, a GP should usually remain the first baseline.
